In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/balanced.csv')


df['timestamp'] = (
    pd.to_datetime('2024' + df['DE13_local_date'].astype(str).str.zfill(4), format='%Y%m%d')
    + pd.to_timedelta(df['hour_local'], unit='h')
)


df = df.sort_values(['client_id', 'timestamp']).set_index('timestamp')



int64
[317 327 531 213 603 329 623 227 121 101 611 503 411 501 619 128 608 504
 405 310]


In [ ]:
# variable time_since_last_txn_min ───────────────────────────────────────────────
df['time_since_last_txn_min'] = (
    df.groupby('client_id').apply(lambda g: g.index.to_series().diff().dt.total_seconds() / 60)
    .reset_index(level=0, drop=True)
).fillna(9999)



In [ ]:
# variable txn_count_last_24h ────────────────────────────────────────────────────
df['txn_count_last_24h'] = (
    df.groupby('client_id')['DE4_amount_transaction']
    .transform(lambda x: x.rolling('24h', closed='left').count())
)

In [ ]:

df = df.set_index('timestamp')

df['txn_count_last_1h'] = (
    df.groupby('client_id')['DE4_amount_transaction']
    .transform(lambda x: x.rolling('1h', closed='left').count())
)

df = df.reset_index()

print(df['txn_count_last_1h'].describe())

count    479.000000
mean       1.394572
std        0.889767
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        6.000000
Name: txn_count_last_1h, dtype: float64
